## デコレーターに基づく実装

In [1]:
from typing import Any
from langchain.agents.middleware import before_model, after_model, before_agent, after_agent, AgentMiddleware
from langgraph.runtime import Runtime
from langchain.agents import AgentState


@before_model
def before_model_moddleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----->before_model<-----"
    return None


@after_model
def after_model_moddleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----->after_model<-----"
    return None


@before_agent
def before_agent_moddleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----->before_agent<-----"
    return None


@after_agent
def after_agent_moddleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    state["messages"][-1].content += "----->after_agent<-----"
    return None

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

import os
from http.client import responses

from dataclasses_json import config
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# 環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [3]:
agent = create_agent(
    model=model,
    middleware=[
        before_model_moddleware,
        after_model_moddleware,
        before_agent_moddleware,
        after_agent_moddleware,

    ],
)

responses = agent.invoke({
    "messages": HumanMessage("こんにちは"),
})

for msg in responses["messages"]:
    msg.pretty_print()

================================ Human Message =================================

こんにちは----->before_agent<---------->before_model<-----
================================== Ai Message ==================================

こんにちは！  
何かお手伝いできますか？----->after_model<---------->after_agent<-----


## クラスに基づく実装

In [4]:
class MyMiddleware(AgentMiddleware):
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----->before_model<-----"
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----->after_model<-----"
        return None

    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----->before_agent<-----"
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        state["messages"][-1].content += "----->after_agent<-----"
        return None

In [5]:
myMiddleware = MyMiddleware()

agent = create_agent(
    model=model,
    middleware=[myMiddleware],
)

responses = agent.invoke({
    "messages": HumanMessage("こんにちは"),
})

for msg in responses["messages"]:
    msg.pretty_print()

================================ Human Message =================================

こんにちは----->before_agent<---------->before_model<-----
================================== Ai Message ==================================

こんにちは。  
どうしましたか？----->after_model<---------->after_agent<-----


## 2 デコレーターパラメータ：can_jump_to
### デコレーターに基づく実装

In [6]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import before_model, after_model, AgentState
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime


@tool
def get_news() -> str:
    """当日のニュースを取得する"""
    return f"北中米共催ワールドカップが本日開幕"


# モデル（LLM）実行前にトリガーされる。"tools" ノードへのジャンプを許可する。
@before_model(can_jump_to=["tools"])
def force_tool_first(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【業務シナリオ：強制インターセプトしてツールをトリガー】
    ユーザー入力に "direct tool" が含まれる場合、今回の大規模言語モデルの思考/生成段階をスキップし、
    直接大規模言語モデルの tool_calls 意図を偽造して、強制的に制御権をツール実行ノードに渡す。
    """
    text = state["messages"][-1].content
    # キーワードをチェックし、条件を満たせば強制的にフローに介入する
    if isinstance(text, str) and "direct tool" in text.lower():
        print("[MIDDLEWARE] before_model: jump_to='tools'")

        # 大規模言語モデルのメッセージオブジェクト（AIMessage）を人工的に構築する
        # システムを欺き、モデル自身が呼び出しを決定したツールであるとシステムに誤認させる
        fake_tool_call = AIMessage(
            content="人工的に構築されたメッセージ",
            tool_calls=[
                {
                    "name": "get_news",
                    "args": {},
                    "id": "call_force_weather_001",
                }
            ],
        )

        # 更新後の状態を返す：偽造したメッセージを注入し、次に "tools" ノードへジャンプすることを明示的に指定する
        return {
            "messages": [fake_tool_call],
            "jump_to": "tools",
        }
    # トリガー条件を満たさない場合は None を返し、フローは正常に下流へ進む（LLM に思考を続けさせる）
    return None

# モデル（LLM）が生成を実行した後にトリガーされる。"model" ノードへの再ジャンプを許可する。
@after_model(can_jump_to=["model"])
def retry_with_extra_instruction(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【業務シナリオ：内省/リトライメカニズム】
    大規模言語モデルがすでに回答を生成しているが、ユーザーの最初のリクエストに "retry model" が含まれていることが分かった場合、
    システムプロンプト（SystemMessage）を動的に追加し、モデルに強制的にもう一度生成（リトライ）させる。
    """
    # メッセージ履歴を逆順に走査し、直近のユーザー入力（human メッセージ）を見つける
    user_text = ""
    for msg in reversed(state["messages"]):
        if getattr(msg, "type", "") == "human":
            user_text = getattr(msg, "content", "")
            break

    # ユーザー入力にリトライをトリガーするキーワードが含まれているか確認する
    if isinstance(user_text, str) and "retry model" in user_text.lower():
        # 【コア防御】：無限ループの再ジャンプ（デッドロック）を防ぐ
        # メッセージ履歴にこの特殊なシステムプロンプトがすでに注入されているか確認する。もしあれば、すでにリトライ済みであることを意味し、再度介入しない。
        already_injected = any(
            isinstance(getattr(msg, "content", None), str)
            and "あなたは必ず【二回目の回答】で始めてください" in msg.content
            for msg in state["messages"]
        )
        if already_injected:
            return None # すでに注入済み。そのまま通過させ、リトライフローを終了する

        print("[MIDDLEWARE] after_model: jump_to='model' with extra system instruction")

        # 更新後の状態を返す：強力な制約を持つシステムメッセージを追加し、ポインタを "model" ノードに戻して再実行する
        return {
            "messages": [
                SystemMessage("あなたは必ず【二回目の回答】で始めてください。そして一文だけで回答してください。")
            ],
            "jump_to": "model",
        }

    return None

# モデル（LLM）実行前にトリガーされる。"end" ノードへの直接ジャンプ（強制終了）を許可する。
@before_model(can_jump_to=["end"])
def overflow_context_processor(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    【業務シナリオ：セーフガード/例外インターセプト】
    コンテキストウィンドウのオーバーフロー（トークン超過）や、その他の深刻なシステムブロッキング状況をシミュレートする。
    一度トリガーされると、フローを直ちに遮断し、大規模言語モデルにこれ以上処理させず、直接エラーを出すかフォールバック文言を返す。
    """

    # 溢出のふりをする。最後のメッセージに overflow マーカーが含まれているかをチェックすることで模擬する
    if "overflow" in state["messages"][-1].content:
        print("[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow")

        # フォールバックの終了メッセージを構築し、直接 "end" へのジャンプを指定して Agent の実行を終了する
        return {
            "messages": [
                AIMessage("コンテキストウィンドウがオーバーフローしました。終了します")
            ],
            "jump_to": "end",
        }


agent = create_agent(
    model=model,
    tools=[get_news],
    # # 定義したミドルウェアを順番に Agent にマウントする（注意：実行順序はリストの宣言順に厳密に従う）
    middleware=[force_tool_first, retry_with_extra_instruction, overflow_context_processor],
)


def run_once(user_input: str):
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": user_input}
            ]
        }
    )

    for msg in result["messages"]:
        msg.pretty_print()


if __name__ == "__main__":
    # Case 1: tools へ直接ジャンプ
    # 期待される動作：
    # 1. force_tool_first がトリガーされ、"[MIDDLEWARE] before_model: jump_to='tools'" を出力
    # 2. LLM の最初の思考をバイパスし、直接 `get_news` ツールを呼び出す
    # 3. ツールが結果を返した後、LLM がツールの結果を要約して出力する
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("今日のニュースを調べてください direct tool")

    # Case 2: 出力後 model へジャンプして戻る
    # 期待される動作：
    # 1. 正常に LLM に入り、1回目の回答を生成
    # 2. retry_with_extra_instruction がトリガーされ、"[MIDDLEWARE] after_model: jump_to='model'..." を出力
    # 3. システムプロンプトが注入された後、LLM が強制的に引き戻され、2回目の回答を生成
    # 4. 最終出力には“【二回目の回答】”というプレフィックスが付いているはず
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("LangChain について適当に紹介してください retry model")

    # Case 3:
    # 期待される動作：
    # 1. overflow_context_processor ミドルウェアがトリガーされる
    # 2. 終了メッセージを直接出力して終了する。LLM はこのリクエストを一切受け取らない
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("こんにちは overflow")

    # Case 4: 通常フロー
    # 期待される動作：
    # 1. どのミドルウェアもトリガーされない（どのキーワードも満たさない）
    # 2. Agent は通常の OOTB（Out of the box）標準ワークフローを実行する：User -> Model -> Call Tool -> Model -> End
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日のニュース概要は？")

============================== -> Case 1 <- ==============================

[MIDDLEWARE] before_model: jump_to='tools'

================================ Human Message =================================

今日のニュースを調べてください direct tool
================================== Ai Message ==================================

人工的に構築されたメッセージ
Tool Calls:
  get_news (call_force_weather_001)
 Call ID: call_force_weather_001
  Args:
================================= Tool Message =================================
Name: get_news

北中米共催ワールドカップが本日開幕
================================== Ai Message ==================================

本日のニュースはこちらです。

- 北中米共催ワールドカップが本日開幕


============================== -> Case 2 <- ==============================

[MIDDLEWARE] after_model: jump_to='model' with extra system instruction

...

============================== -> Case 3 <- ==============================

[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow

================================ Human Message =================================

こんにちは overflow
================================== Ai Message ==================================

コンテキストウィンドウがオーバーフローしました。終了します


============================== -> Case 4 <- ==============================

================================ Human Message =================================

今日のニュース概要は？
================================== Ai Message ==================================
Tool Calls:
  get_news (call_gPd7zilkuovXluJCFYvGyeZ3)
 Call ID: call_gPd7zilkuovXluJCFYvGyeZ3
  Args:
================================= Tool Message =================================
Name: get_news

北中米共催ワールドカップが本日開幕
================================== Ai Message ==================================

本日のニュース概要です。

- **北中米共催ワールドカップが本日開幕**

必要なら、このニュースをもう少し詳しく要約してお伝えできます。


### 2.2 クラスに基づく実装

In [7]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import hook_config, AgentState, AgentMiddleware
from langchain.messages import AIMessage, SystemMessage
from langchain.tools import tool
from langgraph.runtime import Runtime


@tool
def get_news() -> str:
    """当日のニュースを取得する"""
    return f"北中米共催ワールドカップが本日開幕"


class MyMiddleware(AgentMiddleware):
    @hook_config(can_jump_to=["tools", "end"])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        text = state["messages"][-1].content
        # 溢出のふり
        if "overflow" in text:
            print("[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow")
            return {
                "messages": [
                    AIMessage("コンテキストウィンドウがオーバーフローしました。終了します")
                ],
                "jump_to": "end",
            }

        if isinstance(text, str) and "direct tool" in text.lower():
            print("[MIDDLEWARE] before_model: jump_to='tools'")

            fake_tool_call = AIMessage(
                content="人工的に構築されたメッセージ",
                tool_calls=[
                    {
                        "name": "get_news",
                        "args": {},
                        "id": "call_force_weather_001",
                    }
                ],
            )

            return {
                "messages": [fake_tool_call],
                "jump_to": "tools",
            }

        return None

    @hook_config(can_jump_to=["model"])
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        user_text = ""
        for msg in reversed(state["messages"]):
            if getattr(msg, "type", "") == "human":
                user_text = getattr(msg, "content", "")
                break

        if isinstance(user_text, str) and "retry model" in user_text.lower():
            # 無限再ジャンプを防止：すでにプロンプトを追加済みなら、もうジャンプしない
            already_injected = any(
                isinstance(getattr(msg, "content", None), str)
                and "あなたは必ず【二回目の回答】で始めてください" in msg.content
                for msg in state["messages"]
            )
            if already_injected:
                return None

            print("[MIDDLEWARE] after_model: jump_to='model' with extra system instruction")

            return {
                "messages": [
                    SystemMessage("あなたは必ず【二回目の回答】で始めてください。そして一文だけで回答してください。")
                ],
                "jump_to": "model",
            }

        return None


agent = create_agent(
    model=model,
    tools=[get_news],
    middleware=[MyMiddleware()],
)


def run_once(user_input: str):
    result = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": user_input}
            ]
        }
    )

    for msg in result["messages"]:
        msg.pretty_print()


if __name__ == "__main__":
    # Case 1: tools へ直接ジャンプ
    print('=' * 30, '-> Case 1 <-', '=' * 30)
    run_once("今日のニュースを調べてください direct tool")

    # Case 2: 出力後 model へジャンプして戻る
    print('=' * 30, '-> Case 2 <-', '=' * 30)
    run_once("LangChain について適当に紹介してください retry model")

    # Case 3:
    print('=' * 30, '-> Case 3 <-', '=' * 30)
    run_once("こんにちは overflow")

    # Case 4: 通常フロー
    print('=' * 30, '-> Case 4 <-', '=' * 30)
    run_once("今日のニュース概要は？")

============================== -> Case 1 <- ==============================

[MIDDLEWARE] before_model: jump_to='tools'

================================ Human Message =================================

今日のニュースを調べてください direct tool
================================== Ai Message ==================================

人工的に構築されたメッセージ
Tool Calls:
  get_news (call_force_weather_001)
 Call ID: call_force_weather_001
  Args:
================================= Tool Message =================================
Name: get_news

北中米共催ワールドカップが本日開幕
================================== Ai Message ==================================

本日のニュースです。

- 北中米共催ワールドカップが本日開幕

必要なら、このニュースをもとに「詳しい要約」や「関連トピック」も整理できます。


============================== -> Case 2 <- ==============================

[MIDDLEWARE] after_model: jump_to='model' with extra system instruction

...

============================== -> Case 3 <- ==============================

[MIDDLEWARE] before_model: jump_to='end' when contenxt window overflow

================================ Human Message =================================

こんにちは overflow
================================== Ai Message ==================================

コンテキストウィンドウがオーバーフローしました。終了します


============================== -> Case 4 <- ==============================

================================ Human Message =================================

今日のニュース概要は？
================================== Ai Message ==================================
Tool Calls:
  get_news (call_L3c3lyaxQWksr1DYLGEGmzgt)
 Call ID: call_L3c3lyaxQWksr1DYLGEGmzgt
  Args:
================================= Tool Message =================================
Name: get_news

北中米共催ワールドカップが本日開幕
================================== Ai Message ==================================

本日のニュース概要です。

- **北中米共催ワールドカップが本日開幕**

必要であれば、このニュースの**詳しい内容**や**関連する注目ポイント**もまとめます。
